In [1]:
from sidrapy import get_table
import pandas as pd
from tqdm import tqdm
import requests

# Baixar lista de municípios do RS (códigos IBGE)
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios"
municipios_rs = requests.get(url).json()

# Extrair os códigos
rs_codes = [str(m["id"]) for m in municipios_rs]
rs_codes_str = ",".join(rs_codes)

anos = ["2010","2011","2012","2013","2014","2015","2016","2017","2018","2019","2020","2021","2022", "2023", "2024"]
dfs = []

total = len(rs_codes) * len(anos)

with tqdm(total=total) as pbar:
    for cod in rs_codes:
        for ano in anos:
            try:
                df = get_table(
                    table_code="1612",
                    territorial_level="6",          
                    ibge_territorial_code=cod,      
                    period=ano,             
                    variable="109,216,214,112,215",
                    classifications={"81": "all"}
                )
                dfs.append(df)
            except Exception as e:
                print(f"Erro no município {cod}: {e}")
            finally:
                pbar.update(1)

# Concatenar todos os resultados
final = pd.concat(dfs, ignore_index=True)



100%|██████████| 7455/7455 [1:21:26<00:00,  1.53it/s]


In [2]:
df

,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Ano (Código),Ano,Variável (Código),Variável,Produto das lavouras temporárias (Código),Produto das lavouras temporárias
1,6,Município,1006,Hectares,371,4323804,Xangri-lá (RS),2024,2024,109,Área plantada,0,Total
2,6,Município,1006,Hectares,-,4323804,Xangri-lá (RS),2024,2024,109,Área plantada,2688,Abacaxi*
3,6,Município,1006,Hectares,...,4323804,Xangri-lá (RS),2024,2024,109,Área plantada,40471,Alfafa fenada
4,6,Município,1006,Hectares,-,4323804,Xangri-lá (RS),2024,2024,109,Área plantada,2689,Algodão herbáceo (em caroço)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,6,Município,40,Mil Reais,-,4323804,Xangri-lá (RS),2024,2024,215,Valor da produção,2713,Soja (em grão)
167,6,Município,40,Mil Reais,-,4323804,Xangri-lá (RS),2024,2024,215,Valor da produção,2714,Sorgo (em grão)
168,6,Município,40,Mil Reais,-,4323804,Xangri-lá (RS),2024,2024,215,Valor da produção,2715,Tomate
169,6,Município,40,Mil Reais,-,4323804,Xangri-lá (RS),2024,2024,215,Valor da produção,2716,Trigo (em grão)


In [ ]:
# Concatenar todos os resultados
final = pd.concat(dfs, ignore_index=True)

final = final.drop(columns=['NC', 'NN', 'MC', 'D1N', 'D2C', 'D3C', 'D4C'])

final.columns = final.iloc[0] 
final = final[1:]              
final = final.reset_index(drop=True)

final = final.rename(columns={
    "Valor": "valor",
    "Município (Código)": "cod_municipio",
    "Ano": "ano",
    "Variável": "variavel",
    "Produto das lavouras temporárias": "produto"
})
final = final[final["valor"] != "-"]

final["valor"] = pd.to_numeric(final["valor"], errors="coerce")
final = final.dropna(subset=["valor"])

final["ano"] = final["ano"].astype(int)

final["produto"] = (
    final["produto"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.replace("*", "", regex=False)
    .str.strip()
)

final = final[final["produto"].isin(["Soja", "Milho", "Trigo", "Fumo"])]

display(final)

,Unidade de Medida,valor,cod_municipio,ano,variavel,produto
27,Hectares,1500.0,4300034,2010,Área plantada,Milho
29,Hectares,2100.0,4300034,2010,Área plantada,Soja
32,Hectares,700.0,4300034,2010,Área plantada,Trigo
61,Hectares,1500.0,4300034,2010,Área colhida,Milho
63,Hectares,2100.0,4300034,2010,Área colhida,Soja
...,...,...,...,...,...,...
1274492,Hectares,240.0,4323804,2023,Área plantada,Soja
1274526,Hectares,240.0,4323804,2023,Área colhida,Soja
1274560,Toneladas,864.0,4323804,2023,Quantidade produzida,Soja
1274594,Quilogramas por Hectare,3600.0,4323804,2023,Rendimento médio da produção,Soja


In [6]:
final.to_csv(r"..\data\dados_producao_agricola\dados_producao_noroeste_RS.csv", index=False)

print("✅ Dados geográficos tratados com sucesso!")

✅ Dados geográficos tratados com sucesso!


In [77]:
display(final)

,Unidade de Medida,valor,cod_municipio,ano,variavel,produto
27,Hectares,1500.0,4300034,2010,Área plantada,Milho
29,Hectares,2100.0,4300034,2010,Área plantada,Soja
32,Hectares,700.0,4300034,2010,Área plantada,Trigo
61,Hectares,1500.0,4300034,2010,Área colhida,Milho
63,Hectares,2100.0,4300034,2010,Área colhida,Soja
...,...,...,...,...,...,...
84843,Hectares,10.0,4323804,2010,Área plantada,Milho
84877,Hectares,10.0,4323804,2010,Área colhida,Milho
84911,Toneladas,16.0,4323804,2010,Quantidade produzida,Milho
84945,Quilogramas por Hectare,1600.0,4323804,2010,Rendimento médio da produção,Milho


In [ ]:

# remove cabeçalho duplicado
df = df.iloc[1:]
df.columns = df.iloc[0]
df = df[1:]

df = df.reset_index(drop=True) 



In [ ]:
import pandas as pd
import io

arquivo  = r'C:\Users\Bruno\OneDrive - PUCRS - BR\Documentos\Projeto-Em-Business-Intelligence-e-Analytics\data\dados_producao_agricola\dados_extraidos\dados_ibge_producao_2024.csv'

# Ler todas as linhas
with open(arquivo, encoding="utf-8") as f:
    linhas = f.readlines()


In [32]:
blocos = []
variavel = None
buffer = []

for linha in linhas:
    # Detecta nova variável
    if "Variável -" in linha:
        variavel = linha.split("Variável -")[-1].strip()
    
    # Detecta cabeçalho de dados (linha que contém "Total;Milho")
    elif "Total" in linha and "Milho" in linha:
        buffer = [linha]  # inicia novo bloco
    
    # Linhas de dados (começam com número e têm ;)
    elif buffer and ";" in linha and linha[0].isdigit():
        buffer.append(linha)
    
        # Se chegamos ao fim de um bloco (próxima linha não é dado), processa
        # Aqui simplificado: processa sempre que acumulamos dados
        if variavel and buffer:
            df = pd.read_csv(io.StringIO("\n".join(buffer)), sep=";")
            df["Variavel"] = variavel
            blocos.append(df)
            buffer = []  # limpa para próximo bloco

# Junta tudo
final = pd.concat(blocos, ignore_index=True)

ValueError: No objects to concatenate

In [ ]:
arquivo  = r'C:\Users\Bruno\OneDrive - PUCRS - BR\Documentos\Projeto-Em-Business-Intelligence-e-Analytics\data\dados_producao_agricola\dados_extraidos\dados_ibge_producao_2024.csv'

# Ler todas as linhas
with open(arquivo, encoding="utf-8") as f:
    linhas = f.readlines()

blocos = []
variavel = None
buffer = []

for linha in linhas:
    linha = linha.strip()
    
    # Detecta nova variável
    if linha.startswith("Variável -"):
        # Se já havia um buffer, processa
        if buffer:
            df = pd.read_csv(io.StringIO("\n".join(buffer)), sep=";")
            df["Variavel"] = variavel
            blocos.append(df)
            buffer = []
        variavel = linha.replace("Variável - ", "")
    
    # Só adiciona linhas que têm pelo menos 7 colunas separadas por ";"
    elif ";" in linha and len(linha.split(";")) >= 7:
        buffer.append(linha)

# Último bloco
if buffer:
    df = pd.read_csv(io.StringIO("\n".join(buffer)), sep=";")
    df["Variavel"] = variavel
    blocos.append(df)

# Junta tudo
final = pd.concat(blocos, ignore_index=True)

display(final
        )

,#,Cód.,Município,Total,Milho (em grão),Soja (em grão),Trigo (em grão),Variavel
0,1,3165602,Senador Cortes (MG),2,-,-,-,None
1,2,3302452,Macuco (RJ),2,-,-,-,None
2,3,3156205,Rochedo de Minas (MG),6,3,-,-,None
3,4,3204203,Piúma (ES),6,-,-,-,None
4,5,3122900,Dona Euzébia (MG),8,5,-,-,None
...,...,...,...,...,...,...,...,...
6399,1276,5213103,Mineiros (GO),2030408,583428,792142,-,None
6400,1277,3170107,Uberaba (MG),2331644,106437,580261,17556,None
6401,1278,3149804,Perdizes (MG),2413375,148666,269100,36304,None
6402,1279,5211909,Jataí (GO),3841676,1429459,2005988,750,None
